# HWP5 正式记录段落解析修复

依据 Hancom HWP5 记录结构读取源附件；先通过2021/22官方控制稿，再检查2018–2020候选表格行。

In [1]:
import hashlib,json,zlib
from pathlib import Path
import olefile
root=Path.cwd();card=root/'research/kcs_early20_2018_2020_oct_original_attachments_20261023';repair=json.loads((card/'hwp_record_repair_plan.json').read_text());parent=json.loads((card/'hwp_numeric_review_plan.json').read_text());source=json.loads((card/'result.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert repair['frozen'] and olefile.__version__=='0.47' and source['NAS_readback_verified'] and not (card/'hwp_record_repair_result.json').exists()
raw=root/'data/snapshots/kcs_early20_2018_2020_oct_original_attachments_20261023';controls={2021:root/'data/snapshots/kcs_early20_2021_oct_original_source_20261023/2021_early20.hwp',2022:root/'data/snapshots/kcs_early20_2022_oct_original_hwp_20261023/2022_early20.hwp'}
for year,digest in parent['source_HWP_hashes'].items():assert sha(raw/(year+'_attachment.hwp'))==digest
def parse(path):
    with olefile.OleFileIO(path) as doc:
        assert doc.exists('BodyText/Section0');compressed=doc.openstream('BodyText/Section0').read();body=zlib.decompress(compressed,-15);streams=['/'.join(x) for x in doc.listdir()]
    offset=0;records=0;paras=[];extended=0
    while offset+4<=len(body):
        header=int.from_bytes(body[offset:offset+4],'little');tag=header&0x3ff;level=(header>>10)&0x3ff;size=(header>>20)&0xfff;offset+=4
        if size==0xfff:
            assert offset+4<=len(body);size=int.from_bytes(body[offset:offset+4],'little');offset+=4;extended+=1
        assert 0<=size<=len(body)-offset,(records,offset,size,len(body))
        payload=body[offset:offset+size];offset+=size;records+=1
        if tag==67:
            decoded=payload.decode('utf-16le','replace');clean=''.join(ch for ch in decoded if ch.isprintable() or ch==' ');clean=' '.join(clean.split())
            paras.append({'record':records,'level':level,'text':clean[:500]})
    assert len(body)-offset<4
    hits=[]
    for index,p in enumerate(paras):
        if '반도체' in p['text']:
            around=[{'paragraph_index':i,**paras[i]} for i in range(max(0,index-8),min(len(paras),index+18))]
            hits.append({'paragraph_index':index,'paragraph':p,'context':around})
    return {'decompressed_bytes':len(body),'records':records,'paragraphs':len(paras),'extended_size_records':extended,'OLE_stream_count':len(streams),'section_count':len([x for x in streams if x.startswith('BodyText/Section')]),'semiconductor_hits':hits}
review={str(y):parse(p) for y,p in controls.items()}
control_checks={}
for year,amount,yoy in [(2021,'6,394','23.9'),(2022,'5,573','12.8')]:
 hits=review[str(year)]['semiconductor_hits'];matches=[]
 for h in hits:
  following=[z['text'] for z in h['context'] if z['paragraph_index']>h['paragraph_index']]
  if amount in following and any(yoy in z for z in following):matches.append(h['paragraph_index'])
 control_checks[str(year)]={'expected_amount':amount,'expected_yoy':yoy,'matching_semiconductor_paragraph_indices':matches,'passed':bool(matches)}
controls_pass=all(x['passed'] for x in control_checks.values())
if controls_pass:
    for year in (2018,2019,2020):review[str(year)]=parse(raw/(str(year)+'_attachment.hwp'))
result={'decision':'HWP_record_parser_controls_passed_unknown_year_rows_require_semantic_review' if controls_pass else 'HWP_record_parser_control_failed_stop','plan_sha256':sha(card/'hwp_record_repair_plan.json'),'controls':control_checks,'controls_pass':controls_pass,'documents':review,'source_HWP_reads':3 if controls_pass else 0,'control_HWP_reads':2,'external_GETs':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_first_public_bytes_qualified':False,'full_monthly_training_panel_qualified':False}
(card/'hwp_record_repair_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':result['decision'],'controls':control_checks,'unknown_semiconductor_hits':{y:[{'paragraph_index':h['paragraph_index'],'text':h['paragraph']['text'],'next':[(z['paragraph_index'],z['text']) for z in h['context'] if z['paragraph_index']>h['paragraph_index']][:8]} for h in review[y]['semiconductor_hits']] for y in ('2018','2019','2020') if y in review}},ensure_ascii=False))


{"decision": "HWP_record_parser_controls_passed_unknown_year_rows_require_semantic_review", "controls": {"2021": {"expected_amount": "6,394", "expected_yoy": "23.9", "matching_semiconductor_paragraph_indices": [71, 86], "passed": true}, "2022": {"expected_amount": "5,573", "expected_yoy": "12.8", "matching_semiconductor_paragraph_indices": [94], "passed": true}}, "unknown_semiconductor_hits": {"2018": [{"paragraph_index": 75, "text": "ㅇ (주요품목) 전년 동기 대비 반도체(9.4%), 석유제품(108.6%), 승용차(88.7%) 등은 증가, 선박(△61.9%), 무선통신기기(△7.4%) 등은 감소", "next": [[76, "ㅇ (주요국가) 중국(20.6%), 미국(69.6%), EU(46.7%), 베트남(17.3%) 등은 증가, 홍콩(△19.9%), 싱가포르(△3.9%) 등은 감소"], [77, "수입현황"], [78, "ㅇ (주요품목) 전년 동기 대비 원유(58.2%), 가스(70.0%), 반도체 제조용 장비(20.1%), 승용차(6.5%) 등은 증가, 반도체(△0.3%)는 감소"], [79, "ㅇ (주요국가) 중국(28.5%), 중동(25.2%), 미국(87.0%), 일본(15.8%), EU(25.0%), 베트남(42.0%) 등 주요국가 대부분 증가"], [80, "※ 주요품목은 관세청 성질별 분류체계 기준으로 집계되며, 신고수리일 기준으로 연간 통계확정 시(2019년 2월)까지 일부 수치는 정정될 수 있습니다."]]}, {"paragraph_index": 78, "text": "ㅇ (주요품목) 전년 동기 대비 

# 2018–2020 年韩国关税署 10 月 1–20 日原稿附件

三个已定位的关税署官网页面（2018-10-22、2019-10-21、2020-10-21）各有一份按事前规则选中的 HWP 原稿。六次官网请求、三页与三附件的确切原始字节及收取时间、SHA-256 已进入 Yanxuan NAS，远端哈希和限定冷恢复通过。

按 [Hancom 官方 HWP5 记录结构](https://cdn.hancom.com/link/docs/%ED%95%9C%EA%B8%80%EB%AC%B8%EC%84%9C_%ED%8C%8C%EC%9D%BC%ED%98%95%EC%8B%9D_5.0_revision1.2.pdf) 做一次机械段落解析，先在 2021/2022 已验官方控制稿中准确找回半导体出口 **6,394／+23.9%** 与 **5,573／−12.8%**。同一解析在 2018–2020 原稿的出口摘要段落分别读到半导体同比 **+9.4%、−28.8%、+12.1%**；这三份捕获的 HWP **可解析正文**未呈现后来两年那种逐品类金额表，嵌入图片未做数值认证，因此**半导体出口金额保持未知**，不拿其他年份的同比或总出口额倒推。

这些稿件对总出口列出的当期工作日分别为 **14.5、13.5、12**（上年同段为 10、14.5、13.5）。工作日变化不能直接当作半导体自身按日调整系数。五个 10 月官方发布入口及 2018–2022 对应附件现可审计，但只构成五个年度的一个月份，且今日可取的附件字节没有独立首次公开／修订收据。它**不是完整历史训练面、合格 PIT 因子或 SOXX 超额证据**。本卡 0 SOXX 结果读取、0 拟合、0 账户；任何后续信号必须先决定只用官方报告同比还是另找原始金额，并补足每月版本与决策时钟。
